# Dynamic World land-cover download - Colab version


In [ ]:
# Colab setup
!pip -q install geemap==0.32.0 pycrs pyshp

In [ ]:
from __future__ import annotations

import glob
import logging
import os
import shutil
import time
from datetime import datetime, timedelta, date
from pathlib import Path

import ee
import geemap
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

## 1. Mount Google Drive and authenticate Earth Engine

Use the same Google Cloud project ID that you registered for Earth Engine.


In [ ]:
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
PROJECT_ID = 'aiv-gee'

try:
    if PROJECT_ID:
        ee.Initialize(project=PROJECT_ID)
    else:
        ee.Initialize()
except Exception:
    ee.Authenticate()
    if PROJECT_ID:
        ee.Initialize(project=PROJECT_ID)
    else:
        ee.Initialize()

print('Earth Engine initialized')

## 2. Parameters

The defaults below mirror your current Google Drive setup and run a small test first.


In [ ]:
LAND_COVER_BANDS = [
    'water',
    'trees',
    'grass',
    'flooded_vegetation',
    'crops',
    'shrub_and_scrub',
    'built',
    'bare',
    'snow_and_ice',
]

In [ ]:
# Input and output locations
SHAPEFILE_ROOT = Path('/content/drive/MyDrive/gee-download')
OUTPUT_ROOT = Path('/content/drive/MyDrive/gee-download/dynamic_world_result')
TEMP_DIR = Path('/content/dynamic_world_data')

# Use SHAPEFILE_PATHS if your shapefile is not stored as SHAPEFILE_ROOT/id/id.shp.
# SHAPEFILE_PATHS = [Path('/content/drive/MyDrive/gee-download/EU_100km_fishnet_simple_by_distance/EU_100km_fishnet_simple_by_distance.shp')]
SHAPEFILE_PATHS = None

# Choose shapefiles to run when using SHAPEFILE_ROOT.
# None = all folders under SHAPEFILE_ROOT.
FILE_IDS = ['EU_100km_fishnet_simple_by_distance']
FILE_INDEX_RANGE = None

# Time range, inclusive
START_DATE = '2020-01-01'
END_DATE = '2020-01-31'

ID_COLUMN = 'Id'
SCALE_METERS = 1000
SKIP_EXISTING_OUTPUT = False
SLEEP_SECONDS_BETWEEN_EXPORTS = 0

# Dynamic World probability bands. Values are probabilities from 0 to 1.
LAND_COVER_BANDS = [
    'built',
    'bare',
]

STATISTICS = ['MEAN']

## 3. Helper functions


In [ ]:
def parse_date(value) -> date:
    if isinstance(value, date):
        return value
    for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%Y%m%d', '%Y_%m_%d'):
        try:
            return datetime.strptime(str(value), fmt).date()
        except ValueError:
            pass
    raise ValueError(f'Unparsable date format: {value}')


def last_day_of_month(any_day: date) -> date:
    next_month = any_day.replace(day=28) + timedelta(days=4)
    return next_month - timedelta(days=next_month.day)


def month_ranges(begin, end):
    begin = parse_date(begin)
    end = parse_date(end)
    if begin > end:
        raise ValueError('START_DATE must be earlier than or equal to END_DATE')

    result = []
    current = begin
    while current <= end:
        month_end = min(last_day_of_month(current), end)
        result.append((current, month_end))
        current = month_end + timedelta(days=1)
    return result


def format_date(value, output_format='%Y/%m/%d'):
    return parse_date(value).strftime(output_format)


def safe_name(value):
    return str(value).replace('/', '-').replace(' ', '_').replace(':', '-')


def discover_shapefiles(root: Path, file_ids=None, index_range=None, shapefile_paths=None):
    if shapefile_paths is not None:
        records = []
        for shp_path in shapefile_paths:
            shp_path = Path(shp_path)
            if not shp_path.exists():
                raise FileNotFoundError(f'Shapefile does not exist: {shp_path}')
            records.append((shp_path.stem, shp_path))
        return records

    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'SHAPEFILE_ROOT does not exist: {root}')

    if file_ids is None:
        ids = sorted([p.name for p in root.iterdir() if p.is_dir()])
    else:
        ids = list(file_ids)

    if index_range is not None:
        start, stop = index_range
        ids = ids[start:stop]

    records = []
    missing = []
    for file_id in ids:
        shp_path = root / file_id / f'{file_id}.shp'
        if shp_path.exists():
            records.append((file_id, shp_path))
        else:
            missing.append(str(shp_path))

    if missing:
        logging.warning('Missing shapefiles:\n%s', '\n'.join(missing))
    if not records:
        raise FileNotFoundError('No shapefiles found. Check SHAPEFILE_ROOT, FILE_IDS, FILE_INDEX_RANGE, or SHAPEFILE_PATHS.')
    return records

In [ ]:
def dynamic_world_collection_for_period(states, band, start_day, end_day):
    collection = (
        ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
        .filter(ee.Filter.date(
            datetime.combine(start_day, datetime.min.time()),
            datetime.combine(end_day + timedelta(days=1), datetime.min.time()),
        ))
        .map(lambda image: image.select([band]))
        .map(lambda image: image.clip(states))
        .map(lambda image: image.reproject(crs='EPSG:4326', scale=SCALE_METERS))
    )
    return collection


def tidy_zonal_statistics(raw_csv, band, month_start, id_column=ID_COLUMN):
    raw = pd.read_csv(raw_csv)
    if raw.empty:
        return pd.DataFrame(columns=['Date', 'Doy', band, id_column])
    if id_column not in raw.columns:
        raise KeyError(f'Expected ID column {id_column!r} in {raw_csv}. Available columns: {list(raw.columns)}')

    stat_columns = [col for col in raw.columns if col != id_column and col.lower() not in {'system:index', '.geo'}]
    if not stat_columns:
        tidy = raw[[id_column]].copy()
        tidy[band] = pd.NA
    else:
        # geemap.zonal_statistics usually returns one value column such as mean, max, sum, etc.
        value_column = stat_columns[0]
        tidy = raw[[id_column, value_column]].copy().rename(columns={value_column: band})

    day = parse_date(month_start)
    tidy.insert(0, 'Date', format_date(day))
    tidy.insert(1, 'Doy', day.strftime('%j'))
    return tidy[['Date', 'Doy', band, id_column]]


def run_one_output(file_id, shp_path, band, statistic, month_list):
    output_dir = OUTPUT_ROOT / file_id
    output_dir.mkdir(parents=True, exist_ok=True)

    output_csv = output_dir / f'{band}_{statistic}_{START_DATE}_to_{END_DATE}.csv'
    if SKIP_EXISTING_OUTPUT and output_csv.exists():
        logging.info('Skip existing output: %s', output_csv)
        return output_csv

    temp_work_dir = TEMP_DIR / safe_name(file_id) / safe_name(band) / safe_name(statistic)
    if temp_work_dir.exists():
        shutil.rmtree(temp_work_dir)
    temp_work_dir.mkdir(parents=True, exist_ok=True)

    states = geemap.shp_to_ee(str(shp_path))
    monthly_frames = []

    for start_day, end_day in month_list:
        collection = dynamic_world_collection_for_period(states, band, start_day, end_day)
        image_count = collection.size().getInfo()
        if image_count == 0:
            logging.warning('No Dynamic World images found: %s %s to %s', band, start_day, end_day)
            continue

        # Original notebook first averaged all Dynamic World images in the month, then ran zonal statistics.
        monthly_image = collection.mean()
        temp_csv = temp_work_dir / f'dynamic_world_{statistic}_{start_day}_to_{end_day}.csv'

        geemap.zonal_statistics(
            monthly_image,
            states,
            str(temp_csv),
            statistics_type=statistic,
            scale=SCALE_METERS,
        )

        tidy = tidy_zonal_statistics(temp_csv, band, start_day, ID_COLUMN)
        monthly_frames.append(tidy)

        if SLEEP_SECONDS_BETWEEN_EXPORTS:
            time.sleep(SLEEP_SECONDS_BETWEEN_EXPORTS)

    if not monthly_frames:
        logging.warning('No data written for %s / %s / %s', file_id, band, statistic)
        return None

    merged = pd.concat(monthly_frames, ignore_index=True)
    merged = merged.rename(columns={band: f'{band}_{statistic}'})
    merged.to_csv(output_csv, index=False)
    return output_csv

## 4. Preview workload


In [ ]:
shapefiles = discover_shapefiles(SHAPEFILE_ROOT, FILE_IDS, FILE_INDEX_RANGE, SHAPEFILE_PATHS)
time_windows = month_ranges(START_DATE, END_DATE)

print(f'Shapefiles: {len(shapefiles)}')
print(f'Month windows: {len(time_windows)}')
print(f'Land-cover bands: {len(LAND_COVER_BANDS)}')
print(f'Statistics: {len(STATISTICS)}')
print(f'Total output CSV targets: {len(shapefiles) * len(LAND_COVER_BANDS) * len(STATISTICS)}')
print('First shapefiles:', [item[0] for item in shapefiles[:5]])
print('First month windows:', time_windows[:3])

## 5. Run Dynamic World download and zonal statistics



In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

written_files = []
failed_jobs = []

for file_id, shp_path in shapefiles:
    logging.info('Running shapefile %s', file_id)
    for band in LAND_COVER_BANDS:
        for statistic in STATISTICS:
            try:
                output_csv = run_one_output(file_id, shp_path, band, statistic, time_windows)
                if output_csv is not None:
                    written_files.append(output_csv)
                    logging.info('Wrote %s', output_csv)
            except Exception as exc:
                logging.exception('Failed: file_id=%s band=%s statistic=%s', file_id, band, statistic)
                failed_jobs.append({
                    'file_id': file_id,
                    'band': band,
                    'statistic': statistic,
                    'error': repr(exc),
                })

print(f'Finished. Wrote {len(written_files)} files. Failed jobs: {len(failed_jobs)}')